# Explainable Brain Tumor Detection Demo

This notebook demonstrates how to load the model modules, train a classifier or segmentation model, and generate explanations for MRI scans.

In [8]:
import os
import shutil

source_dirs = [
    "../data/classification/Training",
    "../data/classification/Testing"
]

dest_tumor = "data/classification/tumor"
dest_no = "data/classification/no_tumor"

os.makedirs(dest_tumor, exist_ok=True)
os.makedirs(dest_no, exist_ok=True)

tumor_classes = ["glioma", "meningioma", "pituitary"]

for base in source_dirs:
    for cls in os.listdir(base):
        path = os.path.join(base, cls)
        
        if cls in tumor_classes:
            for file in os.listdir(path):
                shutil.copy(os.path.join(path, file), dest_tumor)
        
        elif cls == "notumor":
            for file in os.listdir(path):
                shutil.copy(os.path.join(path, file), dest_no)

print("✅ Classification data ready!")

✅ Classification data ready!


In [9]:
from src.preprocessing import set_seed, build_classification_datasets, build_segmentation_datasets
from src.train import get_classifier, UNet
from src.explainability import GradCAM, overlay_heatmap
from src.evaluate import evaluate_classification, evaluate_segmentation

set_seed(42)

print('Modules imported successfully.')

ModuleNotFoundError: No module named 'src'

In [15]:
import os
import shutil

# Correct base path
BASE_PATH = "../data/segmentation/kaggle_3m"

IMG_DST = "../data/segmentation/images"
MASK_DST = "../data/segmentation/masks"

# Create destination folders
os.makedirs(IMG_DST, exist_ok=True)
os.makedirs(MASK_DST, exist_ok=True)

# Check if path exists
if not os.path.exists(BASE_PATH):
    print("❌ Path not found:", BASE_PATH)
    print("Available folders:", os.listdir("../data/segmentation"))
    raise SystemExit

print("✅ Using dataset:", BASE_PATH)

image_count = 0
mask_count = 0

# Loop through patient folders
for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)

            # Only process .tif images
            if not file.endswith(".tif"):
                continue

            # Separate images and masks
            if "_mask" in file:
                shutil.copy(file_path, MASK_DST)
                mask_count += 1
            else:
                shutil.copy(file_path, IMG_DST)
                image_count += 1

print("✅ Segmentation dataset ready!")
print("Total Images:", image_count)
print("Total Masks:", mask_count)

✅ Using dataset: ../data/segmentation/kaggle_3m
✅ Segmentation dataset ready!
Total Images: 3929
Total Masks: 3929


In [14]:
import os
print(os.listdir("../data/segmentation"))

['kaggle_3m', 'lgg-mri-segmentation']


## Recommended commands

Use these commands from the repository root to train and evaluate models.

In [ ]:
!python src/train.py --task classification --data-dir data/classification --epochs 10 --batch-size 16 --output models/classifier.pth
!python src/train.py --task segmentation --image-dir data/segmentation/images --mask-dir data/segmentation/masks --epochs 20 --batch-size 8 --output models/unet.pth
!python src/evaluate.py --task classification --model-path models/classifier.pth --data-dir data/classification